In [85]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



In [100]:
df = pd.read_csv("coursera.csv")
df.head()

,Subject,Title,Institution,Learning Product,Level,Duration,Gained Skills,Rate,Reviews
0,Business,Business Analysis & Process Management,Coursera Project Network,Guided Project,Beginner,Less Than 2 Hours,"Process Analysis, Business Process, Business A...",4.4,6100
1,Business,Getting Started with Microsoft Excel,Coursera Project Network,Guided Project,Intermediate,Less Than 2 Hours,"Microsoft Excel, Excel Formulas, Spreadsheet S...",4.6,11000
2,Business,Financial Markets,Yale University,Course,Beginner,1 - 3 Months,"Investment Banking, Risk Management, Financial...",4.8,30000
3,Business,Investment Risk Management,Coursera Project Network,Guided Project,Intermediate,Less Than 2 Hours,"Investment Management, Risk Management, Financ...",4.4,1800
4,Business,Food & Beverage Management,Università Bocconi,Course,Mixed,1 - 3 Months,"Food and Beverage, Hospitality, Restaurant Man...",4.8,4800


In [87]:
df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (3404, 9)


In [88]:
text_columns = [
    "Subject",
    "Title",
    "Institution",
    "Learning Product",
    "Level",
    "Gained Skills"
]

for col in text_columns:
    df[col] = df[col].fillna("").astype(str)

In [89]:
# 1. Create a text profile for each course
df["course_profile"] = (
    df["Subject"].astype(str) + " " +
    df["Title"].astype(str) + " " +
    df["Institution"].astype(str) + " " +
    df["Learning Product"].astype(str) + " " +
    df["Level"].astype(str) + " " +
    df["Gained Skills"].astype(str)
)

In [90]:
# 2. Convert course profiles into TF-IDF vectors
tfidf = TfidfVectorizer(
    stop_words="english",
    lowercase=True,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(df["course_profile"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (3404, 29209)


In [91]:
cosine_sim = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Shape:", cosine_sim.shape)

# 4. Create an index for course titles
course_indices = pd.Series(
    df.index,
    index=df["Title"].str.lower()
).drop_duplicates()


Similarity Matrix Shape: (3404, 3404)


In [92]:
course_indices = pd.Series(
    df.index,
    index=df["Title"].str.lower()
).drop_duplicates()

In [93]:
def recommend_courses(course_title, n=10):
    course_title = course_title.lower()

    if course_title not in course_indices:
        print("Course not found.")
        return

    idx = course_indices[course_title]

    # Get similarity scores
    similarity_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity, highest first
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the selected course itself
    similarity_scores = [
        item for item in similarity_scores
        if item[0] != idx
    ]

    # Select top N
    top_courses = similarity_scores[:n]

    # Build results
    results = []

    for course_idx, score in top_courses:
        results.append({
            "Title": df.iloc[course_idx]["Title"],
            "Subject": df.iloc[course_idx]["Subject"],
            "Institution": df.iloc[course_idx]["Institution"],
            "Learning Product": df.iloc[course_idx]["Learning Product"],
            "Level": df.iloc[course_idx]["Level"],
            "Duration": df.iloc[course_idx]["Duration"],
            "Rate": df.iloc[course_idx]["Rate"],
            "Reviews": df.iloc[course_idx]["Reviews"],
            "Similarity": round(float(score), 4)
        })

    return pd.DataFrame(results)


In [102]:
def recommend_from_query(query, num_recommendations=10):

    # Convert user query to TF-IDF vector
    query_vector = tfidf.transform([query])

    # Compare query with all courses
    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Get top courses
    top_indices = np.argsort(
        similarity_scores
    )[::-1][:num_recommendations]

    recommendations = df.iloc[top_indices].copy()

    recommendations["Similarity"] = (
        similarity_scores[top_indices]
    )

    return recommendations[
        [
            "Title",
            "Subject",
            "Institution",
            "Learning Product",
            "Level",
            "Duration",
            "Rate",
            "Reviews",
        ]
    ].reset_index(drop=True)

In [103]:
recommend_from_query(
    "MPA",
    10
)

,Title,Subject,Institution,Learning Product,Level,Duration,Rate,Reviews
0,Finalize a Data Science Project,Information Technology,CertNexus,Course,Intermediate,1 - 4 Weeks,4.8,6
1,"Random Models, Nested and Split-plot Designs",Information Technology,Arizona State University,Course,Intermediate,1 - 4 Weeks,4.1,12
2,Machine Learning: Concepts and Applications,Information Technology,The University of Chicago,Course,Intermediate,1 - 3 Months,4.6,66
3,Basic Data Processing and Visualization,Information Technology,University of California San Diego,Course,Intermediate,1 - 3 Months,4.3,46
4,Python Data Products for Predictive Analytics,Information Technology,University of California San Diego,Specialization,Intermediate,3 - 6 Months,4.6,14
5,Data Engineering with Rust,Information Technology,Duke University,Course,Intermediate,1 - 4 Weeks,4.5,1200
6,Recommender Systems: Evaluation and Metrics,Information Technology,University of Minnesota,Course,Mixed,1 - 3 Months,4.3,1200
7,Data Science in Real Life,Information Technology,Johns Hopkins University,Course,Mixed,1 - 4 Weeks,4.5,256
8,Generative AI Applications and Popular Tools,Information Technology,Edureka,Course,Beginner,1 - 3 Months,3.1,7
9,Clinical Data Models and Data Quality Assessments,Information Technology,University of Colorado System,Course,Intermediate,1 - 3 Months,4.4,16
